<a href="https://colab.research.google.com/github/amarylacs/drowning-detection/blob/main/rpi_cam/imx500_model_conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup

In [ ]:
!sudo apt-get -qq update
!sudo apt-get -qq install -y python3.12 python3.12-venv python3.12-dev
!python3.12 -m venv /content/venv312
!/content/venv312/bin/pip install -q --upgrade pip
!/content/venv312/bin/pip install -q ultralytics onnx "imx500-converter[pt]"
!apt-get -qq install -y openjdk-17-jre-headless
print("Environment ready")

#File Upload

In [ ]:
import zipfile, shutil, os

ZIP_PATH = "/content/DATASET_FILE.zip"
EXTRACT_DIR = "/content/dataset"

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

macosx_dir = os.path.join(EXTRACT_DIR, "__MACOSX")
if os.path.exists(macosx_dir):
    shutil.rmtree(macosx_dir)

print("Extracted to", EXTRACT_DIR)
print(os.listdir(EXTRACT_DIR))

In [ ]:
import glob

MODEL_PATH = "/content/WEIGHTS_FILE.pt"

image_count = len(glob.glob(f"{EXTRACT_DIR}/valid/images/*.jpg")) + \
              len(glob.glob(f"{EXTRACT_DIR}/valid/images/*.png"))

if image_count == 0:
    image_count = len(glob.glob(f"{EXTRACT_DIR}/train/images/*.jpg")) + \
                  len(glob.glob(f"{EXTRACT_DIR}/train/images/*.png"))

TARGET_CALIB_IMAGES = 40
fraction = min(1.0, TARGET_CALIB_IMAGES / image_count)

print(f"Found {image_count} images, using fraction={fraction:.4f} (~{TARGET_CALIB_IMAGES} images)")

#Conversion

In [ ]:
script = f'''
import os
os.environ["MPLBACKEND"] = "Agg"

from ultralytics import YOLO
model = YOLO("{MODEL_PATH}")
model.export(
    format="imx",
    data="{EXTRACT_DIR}/data.yaml",
    fraction={fraction},
    device=0
)
'''

with open("/content/run_export.py", "w") as f:
    f.write(script)

print("Script written")

In [ ]:
!MPLBACKEND=Agg /content/venv312/bin/python /content/run_export.py

#Download Folder/packerOut.zip

In [ ]:
!zip -r /content/IMX_500FOLDER.zip /content/IMX_500FOLDER
